In [ ]:
import logging
import numpy as np
improt scanpy as sc
import pandas as pd
import json
import pickle

clustered_embeddings = sc.read_h5ad(snakemake.input.umap_embedding)
clustered_embeddings.obs.set_index("orig_ids", inplace=True)  # needed to allow transfer

In [ ]:
adata = sc.read_h5ad(snakemake.input.read_count_table)
sc.pp.normalize_total(adata, target_sum=1e4)

In [ ]:
# EnrichR terms
with open(snakemake.input.enrichr_terms, "r") as f:
    terms = json.load(f)


In [ ]:
processed_data = np.load(snakemake.input.processed_data, allow_pickle=True)

In [ ]:
# assert that the order of orig_ids matches the one in adata.var.index
assert (processed_data["orig_ids"] == adata.obs.index).all()

In [ ]:
with open(snakemake.input.gene_log1p_normalizers, "rb") as fp:
    gene_log1p_normalizers = pickle.load(fp)
adata.var["log1p_normalizer"] = gene_log1p_normalizers

In [ ]:
# Add the GPT4 data
# cluster_labels = pd.read_csv(snakemake.input.cellwhisperer_labels, index_col=0)["GPT4_generated_labels"]
# cluster_labels.head()

In [ ]:
adata.obsm["X_cellwhisperer_umap"] = clustered_embeddings.obsm["X_umap"]
adata.obsm["transcriptome_embeds"] = processed_data["transcriptome_embeds"]
adata.obs["leiden"] = clustered_embeddings.obs["leiden"]
# adata.obs["cluster_label"] = adata.obs.leiden.apply(lambda c: cluster_labels.loc[int(c)])
# adata.obs.drop(columns=adata.obs.columns[adata.obs.dtypes == np.int64], inplace=True)
adata.uns["dataset_name"] = snakemake.wildcards.dataset
adata.uns["model_name"] = snakemake.wildcards.model
adata.uns["terms"] = terms

In [ ]:
# Remove columns with an extensive number of categories
drop_cols = [c for c in adata.obs.columns if str(adata.obs[c].dtype) == 'category' and len(adata.obs[c].dtype.categories) > snakemake.params.max_categories_filter]
adata.obs.drop(columns=drop_cols, inplace=True)

In [ ]:
if 'normalized' in adata.layers:
    adata.X = adata.layers['normalized']
    logging.warning("Taking the provided `normalized` layer instead of computing log1p")
else:
    sc.pp.log1p(adata)

In [ ]:
for key in (adata.layers.keys()):
    del adata.layers[key]

for key in list(adata.obsp.keys()):
    del adata.obsp[key]

In [ ]:
adata.X = adata.X.astype(np.float32)

In [ ]:
# TODO optionally reduce `var` dimension (e.g. filter by gene names)

In [ ]:
adata.write_h5ad(snakemake.output.adata)